# Data cleaning for lstm

**INPUT**: Stratified train, validation corpora

**OUTPUT**: Preprocessed train, validation, test corpora

| Step | Decision | Status | Comment |
|------|----------|--------|---------|
| Subject and body columns | Merge | Done | Subject + body tags |
| Cleaning | Remove | Done | Empty values,  duplicates, drop date column |
| Header information | keep | Done | Meaningful for this organizations data |
| Remove message length outliers | Done | lower=15 upper=1.5xIQR | Only on train corpora |
| Patterns to replace | Mask token | Done | Revised and done |
| Repeated chars | Unified length | Done | If same char next to eachother > 3 then replace with 2 |

## Input & Setup

### Imports

In [ ]:
import pandas as pd
import re
import csv
import sys
from pathlib import Path

project_root = Path().resolve().parent
src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from config.config import PATHS, REGEXES

### Read corpora

In [ ]:
train = pd.read_csv(PATHS.train_raw, index_col=0)
validation = pd.read_csv(PATHS.validation_raw, index_col=0)
test = pd.read_csv(PATHS.test_raw, index_col=0)

## Steps

### Merge subject and body columns

In [ ]:
train['Message'] = "subject : " + train['Subject'].astype(str) + "\n" + train['Message'].astype(str)
validation['Message'] = "subject : " + validation['Subject'].astype(str) + "\n" + validation['Message'].astype(str)
test['Message'] = "subject : " + test['Subject'].astype(str) + "\n" + test['Message'].astype(str)

### Cleaning

In [ ]:
# Drop Date column (also drop subject, merged into Message)
train.drop(['Date','Subject'], axis=1, inplace=True)
validation.drop(['Date','Subject'], axis=1, inplace=True)
test.drop(['Date','Subject'], axis=1, inplace=True)

# Remove empty values
train.dropna(inplace=True)
validation.dropna(inplace=True)
test.dropna(inplace=True)

# Remove duplicates
train.drop_duplicates(inplace=True)
validation.drop_duplicates(inplace=True)
test.drop_duplicates(inplace=True)

### Remove message length outliers

In [ ]:
msg_lengths = train["Message"].str.len()
q1 = msg_lengths.quantile(0.25)
q3 = msg_lengths.quantile(0.75)

def calculate_scaled_IQR(q1, q3, scaling_factor = 1.5):
    IQR = q3 - q1
    upper_boundary = int(q3 + scaling_factor*IQR)
    lower = int(q1 - scaling_factor*IQR)
    lower_boundary = lower if lower > 0 else 15
    return upper_boundary, lower_boundary

upper, lower = calculate_scaled_IQR(q1,q3)

def remove_message_length_outliers(data, lower, upper):
    mask = data["Message"].str.len().between(lower, upper, inclusive='both')
    data = data[mask]
    return data
    
train = remove_message_length_outliers(train, lower=lower, upper=upper)
validation = remove_message_length_outliers(validation, lower=lower, upper=upper)
test = remove_message_length_outliers(test, lower=lower, upper=upper)   

    

### Replace email, url, phone, num

In [ ]:
def remove_spaces_around_spec_chars(text):
    text = str(text)
    text = re.sub(r" +([:\/?#[\]@!$&'()*+,;=%_.~-])", r"\1", text)
    text = re.sub(r"([:\/?#[\]@!$&'()*+,;=%_.~-]) +", r"\1", text)
    return text

train['Message'] = train['Message'].apply(remove_spaces_around_spec_chars)
validation['Message'] = validation['Message'].apply(remove_spaces_around_spec_chars)
test['Message'] = test['Message'].apply(remove_spaces_around_spec_chars)

In [ ]:

def preserve_email_url_phone_num(message):
    message = str(message)
    message = re.sub(REGEXES.email, '<MAIL>', message)
    message = re.sub(REGEXES.url, '<URL>', message)
    message = re.sub(REGEXES.phone, '<PHONE>', message)
    message = re.sub(REGEXES.num, '<NUM>', message)
    return message

train['Message'] = train['Message'].apply(preserve_email_url_phone_num)
validation['Message'] = validation['Message'].apply(preserve_email_url_phone_num)
test['Message'] = test['Message'].apply(preserve_email_url_phone_num)

### Repeated chars to unified length

In [ ]:
def collapse_repeated_chars_and_spaces(message):
    message = str(message)
    message = re.sub(r'\s+', ' ', message).strip()
    message = re.sub(r'(.)\1{2,}', r'\1\1', message)
    return message

train['Message'] = train['Message'].apply(collapse_repeated_chars_and_spaces)
validation['Message'] = validation['Message'].apply(collapse_repeated_chars_and_spaces)
test['Message'] = test['Message'].apply(collapse_repeated_chars_and_spaces)
    

### Add space after punctuation

In [ ]:
def place_spaces_around_spec_chars(text):
    text = str(text)
    text = re.sub(r"([:\/?#[\]@!$&'()*+,;=%_.~-])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text

train['Message'] = train['Message'].apply(place_spaces_around_spec_chars)
validation['Message'] = validation['Message'].apply(place_spaces_around_spec_chars)
test['Message'] = test['Message'].apply(place_spaces_around_spec_chars)

## Output

### Save lstm preprocessed corpora

In [ ]:
train.to_csv(PATHS.train_processed, index=False, quoting=csv.QUOTE_ALL)
validation.to_csv(PATHS.validation_processed, index=False, quoting=csv.QUOTE_ALL)
test.to_csv(PATHS.test_processed, index=False, quoting=csv.QUOTE_ALL)